# 15 — Full atlas data prep (no holdout, 2 flavors only)

**Pipeline order (training branch):** (1) Load Tabula mouse/human from `.raw` + obs. (2) **BioMart** ortholog alignment on **all** cells (full assay mix). (3) **HSPC merge**: hematopoietic stem (`CL:0000037`) + precursor (`CL:0008001`) → label **`HSPC`** / ontology `CL:0008001` **before** any matching. (4) Match by `(cell_type_ontology_term_id, tissue)` on all assays → `*_no_assay_filter` (§1b only). (5) **Then** subset to mouse **`10x 3' v2`** and human **`10x 3' v3`** to match BCG-style data. (6) Match again → `mouse_matched` / `human_matched`. (7) Concatenate → `matched_full`; **counts** in `layers['counts']` (int32); **`normalize_total` + `log1p`** → **`.X`**. (8) HVG per flavor using **`layer='counts'`**; subset to 1000 genes; `clean_adata` → `hvg_*_atlas_full.h5ad`. (9) Round-trip under CellOT env → **`_v07.h5ad`** (anndata **0.7**-compatible storage for training).

**Two matched pools for comparison (only one is exported).** After one BioMart alignment on all cells, **HSPC**: hematopoietic stem (`CL:0000037`) and precursor (`CL:0008001`) are merged on **`mouse_aln` / `human_aln`** **before** any matching so `(cell_type, tissue)` pooling matches mentor guidance. Then: (1) **no assay filter** — same matching rule as `01.5` on the full matrix (`mouse_matched_no_assay_filter` / …); (2) **current method** — subset to mouse **10x 3' v2** and human **10x 3' v3**, match again → `mouse_matched` / `human_matched`. §1b compares (1) vs (2); **§2 onward uses only (2)**.

Mirror of `01.5_data_prep_all_holdouts_hvg_flavors.ipynb` but with these differences:
- **No per-group holdout loop** — the v2/v3 branch keeps every pair the matcher returns. The trained model is intended for downstream prediction on external data (BCG mouse), so we want it to have seen every cell type during training.
- **Only two HVG flavors**: `seurat_v3` and `pearson_residuals` (per the May 8 mentor meeting). Two trainings × two models = four total downstream training jobs.
- **Output names** use the suffix `_atlas_full` (not per-group letter): `hvg_seurat_v3_atlas_full_v07.h5ad`, `hvg_pearson_residuals_atlas_full_v07.h5ad`.

One alignment; two matches (for §1b vs current); concat + normalization **only** for the v2/v3 branch. Same int32-cast for raw-count flavors; same anndata-0.7 round-trip via the CellOT env.

Biomarker presence is also tracked here (12-marker panel) to feed the meeting figure.

In [1]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp_sparse

sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT")
from speciesot_helpers import (
    align_adatas_biomart_one2one,
    match_cells_by_celltype_tissue,
)

BASE_DIR = "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT"
MOUSE_H5AD = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_muris/sampled_mouse_shared.h5ad"
HUMAN_H5AD = "/n/holylabs/mooney_lab/Lab/joshprice/speciesOT/data/tabula_sapiens/sampled_human_shared.h5ad"

DATASET_DIR = os.path.join(BASE_DIR, "cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg")
os.makedirs(DATASET_DIR, exist_ok=True)

OUT_DIR = os.path.join(BASE_DIR, "speciesOT/baseline/analysis/atlas_full_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

CT_COL = "cell_type_ontology_term_id"
TISSUE_COL = "tissue_ontology_term_id"
N_HVG = 1000
RANDOM_STATE = 0

FLAVORS = ["seurat_v3", "pearson_residuals"]

# Assay targets for the **current** branch only (§1 second match). The no-assay-filter branch ignores these.
MOUSE_ASSAY = "10x 3' v2"
HUMAN_ASSAY = "10x 3' v3"

# HSC + hematopoietic precursor → one label/ontology before match_cells (§1)
HSPC_LABEL = "HSPC"
HSPC_ONTOLOGY = "CL:0008001"  # hematopoietic precursor id; confirm with mentor if preferred
HSPC_MERGE_ONTOLOGIES = frozenset({"CL:0000037", "CL:0008001"})


def merge_hspc_obs(adata, name=""):
    """Collapse HSC + precursor on obs (mutates adata); run on aligned objects before matching."""
    adata.obs["cell_type"] = adata.obs["cell_type"].astype(str)
    adata.obs["cell_type_ontology_term_id"] = (
        adata.obs["cell_type_ontology_term_id"].astype(str)
    )
    m = adata.obs["cell_type_ontology_term_id"].isin(HSPC_MERGE_ONTOLOGIES)
    adata.obs.loc[m, "cell_type"] = HSPC_LABEL
    adata.obs.loc[m, "cell_type_ontology_term_id"] = HSPC_ONTOLOGY
    n = int(m.sum())
    print(f"  HSPC merge [{name}]: {n} rows -> {HSPC_LABEL!r} / {HSPC_ONTOLOGY!r}")
    return n


KEEP_OBS = [
    "condition", "species",
    "cell_type_ontology_term_id", "cell_type",
    "tissue_ontology_term_id", "tissue",
    "donor_id",
]

CELLOT_PY = "/n/home01/jzhou1125/.conda/envs/CellOT/bin/python"

# Curated marker panel (matches notebook 14)
MARKER_PANEL = {
    "PTPRC (CD45)":  "ENSG00000081237",
    "CD3E":          "ENSG00000198851",
    "CD4":           "ENSG00000010610",
    "CD8A":          "ENSG00000153563",
    "CD5":           "ENSG00000110448",
    "CD7":           "ENSG00000173762",
    "CCR7":          "ENSG00000126353",
    "NCAM1 (CD56)":  "ENSG00000149294",
    "MS4A1 (CD20)":  "ENSG00000156738",
    "CD19":          "ENSG00000177455",
    "CD14":          "ENSG00000170458",
    "ITGAM (CD11b)": "ENSG00000169896",
}

sc.settings.verbosity = 1
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("dataset dir:", DATASET_DIR)
print("flavors:", FLAVORS)

/n/home01/jzhou1125/miniforge3/envs/analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


scanpy: 1.12
anndata: 0.12.10
dataset dir: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg
flavors: ['seurat_v3', 'pearson_residuals']


/tmp/ipykernel_4132210/845091797.py:87: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)
/tmp/ipykernel_4132210/845091797.py:88: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)


## 1. Load + align + two matches (comparison)

0. After ortholog alignment on **`mouse_aln` / `human_aln`**: merge **hematopoietic stem cell** (`CL:0000037`) + **hematopoietic precursor** (`CL:0008001`) → label **`HSPC`** with ontology **`HSPC_ONTOLOGY`** *before* any `match_cells` call so both matches pool those types.
1. Match on **`mouse_aln` / `human_aln`** with **no assay filtering** → `mouse_matched_no_assay_filter` / `human_matched_no_assay_filter` (same spirit as `01.5`).
2. Subset to **`MOUSE_ASSAY` / `HUMAN_ASSAY`**, match again → **`mouse_matched` / `human_matched`** (current pipeline; used in §2 onward).

In [2]:
print("Loading source files and promoting .raw to .X ...")
mouse_full = sc.read_h5ad(MOUSE_H5AD)
human_full = sc.read_h5ad(HUMAN_H5AD)
assert mouse_full.raw is not None
assert human_full.raw is not None

mouse_all = mouse_full.raw.to_adata()
human_all = human_full.raw.to_adata()
mouse_all.obs = mouse_full.obs.copy()
human_all.obs = human_full.obs.copy()
mouse_all.X = mouse_all.X.astype("float32")
human_all.X = human_all.X.astype("float32")
print(f"  mouse_all: {mouse_all.shape}")
print(f"  human_all: {human_all.shape}")
print("\n  assay mix (before any filter) — mouse:")
print(mouse_all.obs["assay"].value_counts().to_string())
print("  human:")
print(human_all.obs["assay"].value_counts().to_string())

print("\nAligning orthologs (BioMart one2one) on all cells ...")
mouse_aln, human_aln, ortholog_table = align_adatas_biomart_one2one(mouse_all, human_all)
print(f"  ortholog pairs: {len(ortholog_table)}")
print(f"  mouse_aln: {mouse_aln.shape}")
print(f"  human_aln: {human_aln.shape}")

ortho_cache_path = "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/scripts/.biomart_ortholog_cache.csv"
ortholog_table.to_csv(ortho_cache_path, index=False)
print(f"  cached ortholog table -> {ortho_cache_path}")

print("\nMerge HSC + precursor -> HSPC (before matching) ...")
merge_hspc_obs(mouse_aln, "mouse_aln")
merge_hspc_obs(human_aln, "human_aln")

print("\nMatching — no assay filter (all assays; like 01.5) ...")
mouse_matched_no_assay_filter, human_matched_no_assay_filter = match_cells_by_celltype_tissue(
    mouse_aln, human_aln,
    cell_type_key=CT_COL, tissue_key=TISSUE_COL, seed=RANDOM_STATE,
)
print(f"  mouse_matched_no_assay_filter: {mouse_matched_no_assay_filter.shape}")
print(f"  human_matched_no_assay_filter: {human_matched_no_assay_filter.shape}")

print(f"\nAssay subset aligned adatas → {MOUSE_ASSAY!r} / {HUMAN_ASSAY!r} ...")
mouse_aligned = mouse_aln[mouse_aln.obs["assay"].astype(str) == MOUSE_ASSAY].copy()
human_aligned = human_aln[human_aln.obs["assay"].astype(str) == HUMAN_ASSAY].copy()
print(f"  mouse_aligned: {mouse_aligned.shape}")
print(f"  human_aligned: {human_aligned.shape}")

print("\nMatching (v2/v3 only; main pipeline) ...")
mouse_matched, human_matched = match_cells_by_celltype_tissue(
    mouse_aligned, human_aligned,
    cell_type_key=CT_COL, tissue_key=TISSUE_COL, seed=RANDOM_STATE,
)
print(f"  mouse_matched: {mouse_matched.shape}")
print(f"  human_matched: {human_matched.shape}")

Loading source files and promoting .raw to .X ...
  mouse_all: (47807, 18024)
  human_all: (58931, 61759)

  assay mix (before any filter) — mouse:
assay
10x 3' v2     30326
Smart-seq2    17481
  human:
assay
10x 3' v3     51192
10x 5' v2      4297
Smart-seq2     3405
Smart-seq3       37

Aligning orthologs (BioMart one2one) on all cells ...
  ortholog pairs: 14451
  mouse_aln: (47807, 14451)
  human_aln: (58931, 14451)
  cached ortholog table -> /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/scripts/.biomart_ortholog_cache.csv

Merge HSC + precursor -> HSPC (before matching) ...
  HSPC merge [mouse_aln]: 2000 rows -> 'HSPC' / 'CL:0008001'
  HSPC merge [human_aln]: 1858 rows -> 'HSPC' / 'CL:0008001'

Matching — no assay filter (all assays; like 01.5) ...
  mouse_matched_no_assay_filter: (6495, 14451)
  human_matched_no_assay_filter: (6495, 14451)

Assay subset aligned adatas → "10x 3' v2" / "10x 3' v3" ...
  mouse_aligned: (30326, 14451)
  human_aligned: (51192, 14451)

Matching (v2/v3

In [3]:
mouse_v2 = mouse_matched[mouse_matched.obs['assay']=="10x 3' v2"]
print(mouse_v2.obs['cell_type'].value_counts())


cell_type
HSPC                               891
intermediate monocyte              504
non-classical monocyte             365
natural killer cell                351
vein endothelial cell              278
fibroblast of cardiac tissue       242
thymocyte                          226
bronchial smooth muscle cell       187
endothelial cell                   153
B cell                             153
basophil                           143
plasma cell                        132
pulmonary alveolar type 2 cell     125
large intestine goblet cell         93
T cell                              76
classical monocyte                  74
macrophage                          69
pericyte                            61
monocyte                            36
CD8-positive, alpha-beta T cell     31
smooth muscle cell                  27
plasmacytoid dendritic cell         27
fibroblast                          26
CD4-positive, alpha-beta T cell     25
myeloid dendritic cell               5
neutrophil     

In [4]:
mouse_v2 = mouse_matched[mouse_matched.obs['assay']=="10x 3' v2"]
print(mouse_v2.obs['cell_type'].value_counts())


cell_type
HSPC                               891
intermediate monocyte              504
non-classical monocyte             365
natural killer cell                351
vein endothelial cell              278
fibroblast of cardiac tissue       242
thymocyte                          226
bronchial smooth muscle cell       187
endothelial cell                   153
B cell                             153
basophil                           143
plasma cell                        132
pulmonary alveolar type 2 cell     125
large intestine goblet cell         93
T cell                              76
classical monocyte                  74
macrophage                          69
pericyte                            61
monocyte                            36
CD8-positive, alpha-beta T cell     31
smooth muscle cell                  27
plasmacytoid dendritic cell         27
fibroblast                          26
CD4-positive, alpha-beta T cell     25
myeloid dendritic cell               5
neutrophil     

## 1b. Comparison: no assay filter vs current (v2/v3)

**Stacked counts** = `ad.concat` mouse + human, then `cell_type` value counts (same convention as `matched_full`): **two rows per matched pair**.

In [5]:
def stacked_celltype_counts(mouse_m, human_m):
    return ad.concat([mouse_m, human_m], join="inner").obs["cell_type"].value_counts()

vc_no_filter = stacked_celltype_counts(mouse_matched_no_assay_filter, human_matched_no_assay_filter)
vc_v2v3 = stacked_celltype_counts(mouse_matched, human_matched)

tbl = pd.concat(
    [
        vc_no_filter.rename("stacked_n_no_assay_filter"),
        vc_v2v3.rename("stacked_n_v2_v3_current"),
    ],
    axis=1,
).fillna(0).astype(int)
tbl["delta_current_minus_no_filter"] = tbl["stacked_n_v2_v3_current"] - tbl["stacked_n_no_assay_filter"]
tbl = tbl.sort_values("stacked_n_no_assay_filter", ascending=False)

n_pairs_no_filter = mouse_matched_no_assay_filter.n_obs
n_pairs_v2v3 = mouse_matched.n_obs
print("Matched pairs — no assay filter:", n_pairs_no_filter, " | current (v2/v3):", n_pairs_v2v3)
print("Stacked cells — no filter:", int(vc_no_filter.sum()), " | v2/v3:", int(vc_v2v3.sum()))
print()
print(tbl.to_string())
print()
only_no_filter = tbl.index[tbl["stacked_n_v2_v3_current"] == 0].tolist()
only_v2v3 = tbl.index[tbl["stacked_n_no_assay_filter"] == 0].tolist()
print("Cell types in no-filter pool but empty after v2/v3:", len(only_no_filter))
if only_no_filter:
    print(only_no_filter)
print("Cell types only in v2/v3 pool:", len(only_v2v3))
if only_v2v3:
    print(only_v2v3)

Matched pairs — no assay filter: 6495  | current (v2/v3): 4305
Stacked cells — no filter: 12990  | v2/v3: 8610

                                 stacked_n_no_assay_filter  stacked_n_v2_v3_current  delta_current_minus_no_filter
cell_type                                                                                                         
HSPC                                                  3168                     1782                          -1386
intermediate monocyte                                 1008                     1008                              0
natural killer cell                                    912                      702                           -210
thymocyte                                              910                      452                           -458
non-classical monocyte                                 852                      730                           -122
large intestine goblet cell                            760                      186

## 1c. HSC diagnostic (assay × tissue)

On **`mouse_aln` / `human_aln`** (all assays, post-alignment): where do **hematopoietic stem cell** rows live by assay, and how many shared `(ontology, tissue)` identities still have both mouse v2 and human v3 after subsetting?

In [6]:
HSC_ONTOLOGY = "CL:0000037"  # hematopoietic stem cell (pre-merge; see §1)

for name, ads in [("mouse", mouse_all), ("human", human_all)]:
    sub = ads[ads.obs[CT_COL].astype(str) == HSC_ONTOLOGY]
    print(f"\n=== {name} — HSC n={sub.n_obs} ===")
    if sub.n_obs == 0:
        print("  (no rows with this cell_type string)")
        continue
    print("assay:")
    print(sub.obs["assay"].value_counts().to_string())
    grp = sub.obs.groupby([TISSUE_COL, "assay"], observed=False).size().sort_values(ascending=False).head(25)
    print("\ntop (tissue_ontology_term_id, assay) counts:")
    print(grp.to_string())

mouse_hsc = mouse_all[mouse_all.obs[CT_COL].astype(str) == HSC_ONTOLOGY]
human_hsc = human_all[human_all.obs[CT_COL].astype(str) == HSC_ONTOLOGY]

def _id_keys(adata):
    o = adata.obs[CT_COL].astype(str)
    t = adata.obs[TISSUE_COL].astype(str)
    return set(zip(o, t))

key_m = _id_keys(mouse_hsc)
key_h = _id_keys(human_hsc)
shared = key_m & key_h
print(f"\n=== HSC shared (cell_type_ontology_term_id, tissue_ontology_term_id) identities ===")
print(f"  mouse identities: {len(key_m)}  human: {len(key_h)}  intersection: {len(shared)}")

mouse_v2 = mouse_hsc[mouse_hsc.obs["assay"].astype(str) == MOUSE_ASSAY]
human_v3 = human_hsc[human_hsc.obs["assay"].astype(str) == HUMAN_ASSAY]
k_m2 = _id_keys(mouse_v2)
k_h3 = _id_keys(human_v3)
both_v2v3 = k_m2 & k_h3
print(f"\nAfter assay subset (mouse {MOUSE_ASSAY!r}, human {HUMAN_ASSAY!r}):")
print(f"  identities with mouse HSC v2: {len(k_m2)}  with human HSC v3: {len(k_h3)}")
print(f"  identities with BOTH sides HSC under v2/v3: {len(both_v2v3)}")
lost = shared - both_v2v3
if lost:
    print(f"\nIdentities present in full HSC pool ({len(shared)}) but absent from v2∩v3 on both sides: {len(lost)}")
    for t in list(lost)[:15]:
        print(" ", t)
    if len(lost) > 15:
        print("  ...")
else:
    print("\n(no loss of shared identities from assay filter — check matching / labels if HSC still 0 in §1b)")


=== mouse — HSC n=1000 ===
assay:
assay
Smart-seq2    993
10x 3' v2       7

top (tissue_ontology_term_id, assay) counts:
tissue_ontology_term_id  assay     
UBERON:0002371           Smart-seq2    993
UBERON:0000059           10x 3' v2       7
                         Smart-seq2      0
UBERON:0002371           10x 3' v2       0

=== human — HSC n=858 ===
assay:
assay
10x 3' v3     329
Smart-seq2    271
10x 5' v2     258

top (tissue_ontology_term_id, assay) counts:
tissue_ontology_term_id  assay     
UBERON:0002371           10x 3' v3     278
                         Smart-seq2    262
                         10x 5' v2     252
UBERON:0000178           10x 3' v3      51
                         Smart-seq2      7
                         10x 5' v2       6
UBERON:0002106           Smart-seq2      2
                         10x 5' v2       0
                         10x 3' v3       0

=== HSC shared (cell_type_ontology_term_id, tissue_ontology_term_id) identities ===
  mouse identities: 2

## 2. Concat + store raw counts as int32 + log-normalize

HSPC merge is already applied in §1 **before** matching. Here: **concat** **`mouse_matched` / `human_matched`**, copy raw matrix to **`layers["counts"]`** (int32), then **`normalize_total`** + **`log1p`**.

Uses **`mouse_matched` / `human_matched`** from the **current (v2/v3)** match in §1 — not `mouse_matched_no_assay_filter`.

In [7]:
mouse_m = mouse_matched.copy()
human_m = human_matched.copy()
assert mouse_m.shape[0] == human_m.shape[0]
assert (mouse_m.obs["assay"].astype(str) == MOUSE_ASSAY).all(), mouse_m.obs["assay"].value_counts()
assert (human_m.obs["assay"].astype(str) == HUMAN_ASSAY).all(), human_m.obs["assay"].value_counts()
mouse_m.obs["condition"] = "mouse"
human_m.obs["condition"] = "human"
matched_full = ad.concat([mouse_m, human_m], join="inner")
matched_full.obs["species"] = matched_full.obs["condition"].values

raw_X = matched_full.X
if sp_sparse.issparse(raw_X):
    assert np.allclose(raw_X.data, np.round(raw_X.data))
    raw_int = raw_X.copy(); raw_int.data = raw_int.data.astype(np.int32)
else:
    assert np.allclose(raw_X, np.round(raw_X))
    raw_int = raw_X.astype(np.int32)
matched_full.layers["counts"] = raw_int

sc.pp.normalize_total(matched_full, target_sum=1e4)
sc.pp.log1p(matched_full)

print(f"matched_full: {matched_full.shape}")
print(f"  .X log-norm: min={float(matched_full.X.min()):.4f}, max={float(matched_full.X.max()):.4f}, mean={float(matched_full.X.mean()):.4f}")
counts = matched_full.layers["counts"]
counts_data = counts.data if sp_sparse.issparse(counts) else counts
print(f"  .layers['counts'] dtype: {counts_data.dtype}, min={int(counts_data.min())}, max={int(counts_data.max())}")
print(matched_full)

matched_full: (8610, 14451)
  .X log-norm: min=0.0000, max=8.4648, mean=0.2004
  .layers['counts'] dtype: int32, min=1, max=9536
AnnData object with n_obs × n_vars = 8610 × 14451
    obs: 'free_annotation', 'method', 'donor_id', 'assay_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'is_primary_data', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'shared_cell_type', 'condition', 'species'
    uns: 'log1p'
    obsm: 'X_pca', 'X_umap'
    layers: 'counts'


In [8]:
# sanity: merged HSC+precursor should be labeled HSPC after §1
(matched_full.obs["cell_type"].astype(str) == HSPC_LABEL).sum()

np.int64(1782)

## 3. HVG dispatch + clean_adata helper

In [9]:
def run_hvg_flavor(adata, flavor, n_top=N_HVG, batch_key="species"):
    """Same dispatcher as 01.5; only seurat_v3 / pearson_residuals branches used here."""
    a = adata.copy()
    if "counts" in a.layers:
        L = a.layers["counts"]
        if sp_sparse.issparse(L):
            if L.data.dtype != np.int32:
                L_int = L.copy(); L_int.data = L_int.data.astype(np.int32)
                a.layers["counts"] = L_int
        else:
            if L.dtype != np.int32:
                a.layers["counts"] = L.astype(np.int32)

    if flavor == "seurat_v3":
        sc.pp.highly_variable_genes(
            a, n_top_genes=n_top, flavor="seurat_v3",
            batch_key=batch_key, layer="counts",
        )
        score_col = "variances_norm"
    elif flavor == "pearson_residuals":
        from scanpy.experimental.pp import highly_variable_genes as hvg_pr
        hvg_pr(a, n_top_genes=n_top, flavor="pearson_residuals",
               batch_key=batch_key, layer="counts")
        score_col = "residual_variances" if "residual_variances" in a.var.columns else "highly_variable_rank"
    else:
        raise ValueError(f"Unsupported flavor for this notebook: {flavor!r}")

    df = a.var[["highly_variable"]].copy()
    df["score"] = a.var.get(score_col, np.nan)
    df["flavor"] = flavor
    if "highly_variable_rank" in a.var.columns:
        df["rank"] = pd.to_numeric(a.var["highly_variable_rank"], errors="coerce").astype(float)
    else:
        df["rank"] = df["score"].rank(ascending=False, method="min").where(df["highly_variable"]).astype(float)
    return df


def clean_adata(adata):
    obs_cols = [c for c in KEEP_OBS if c in adata.obs.columns]
    X = adata.X
    if sp_sparse.issparse(X):
        X = np.array(X.todense())
    elif not isinstance(X, np.ndarray):
        X = np.array(X)
    return ad.AnnData(
        X=X.astype(np.float32),
        obs=adata.obs[obs_cols].copy(),
        var=pd.DataFrame(index=adata.var_names),
    )

print("helpers defined")

helpers defined


## 4. Per-flavor HVG selection on the FULL atlas + write 2 datasets

In [10]:
summary_rows = []
biomarker_rows = []

for flavor in FLAVORS:
    print(f"\n{'='*70}\nFLAVOR: {flavor}\n{'='*70}")
    hvg_df = run_hvg_flavor(matched_full, flavor)
    hv_genes = (
        hvg_df[hvg_df["highly_variable"]]
        .sort_values("rank", na_position="last")
        .index.tolist()
    )[:N_HVG]

    # Save full per-flavor ranking
    hvg_df.to_csv(os.path.join(OUT_DIR, f"hvg_atlas_full_{flavor}.csv"))

    # Subset matched_full to the top-1000 HVG
    sub = matched_full[:, hv_genes].copy()
    cleaned = clean_adata(sub)
    out_path = os.path.join(DATASET_DIR, f"hvg_{flavor}_atlas_full.h5ad")
    cleaned.write_h5ad(out_path)
    print(f"  wrote {os.path.basename(out_path)}: {cleaned.n_obs} cells x {cleaned.n_vars} genes")

    # Marker presence in this flavor's top-1000
    selected_set = set(hv_genes)
    for label, ensg in MARKER_PANEL.items():
        in_top1k = ensg in selected_set
        try:
            rank = float(hvg_df.loc[ensg, "rank"])
        except (KeyError, TypeError):
            rank = float("nan")
        biomarker_rows.append({
            "flavor": flavor, "marker": label, "ensg": ensg,
            "in_top1000": in_top1k, "rank": rank,
        })

    summary_rows.append({
        "flavor": flavor,
        "n_hvg": len(hv_genes),
        "n_cells": cleaned.n_obs,
        "n_markers_in_top1000": sum(1 for label in MARKER_PANEL if MARKER_PANEL[label] in selected_set),
        "out_path": out_path,
    })

summary_df = pd.DataFrame(summary_rows)
biomarker_df = pd.DataFrame(biomarker_rows)
summary_df.to_csv(os.path.join(OUT_DIR, "hvg_atlas_full_summary.csv"), index=False)
biomarker_df.to_csv(os.path.join(OUT_DIR, "biomarker_selection_atlas_full.csv"), index=False)

print("\nSummary:"); print(summary_df.to_string(index=False))
print("\nBiomarker pivot:")
print(biomarker_df.pivot_table(index="marker", columns="flavor", values="in_top1000",
                                aggfunc="first").to_string())


FLAVOR: seurat_v3
  wrote hvg_seurat_v3_atlas_full.h5ad: 8610 cells x 1000 genes

FLAVOR: pearson_residuals
  wrote hvg_pearson_residuals_atlas_full.h5ad: 8610 cells x 1000 genes

Summary:
           flavor  n_hvg  n_cells  n_markers_in_top1000                                                                                                                                  out_path
        seurat_v3   1000     8610                     4         /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_seurat_v3_atlas_full.h5ad
pearson_residuals   1000     8610                     7 /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_pearson_residuals_atlas_full.h5ad

Biomarker pivot:
flavor         pearson_residuals  seurat_v3
marker                                     
CCR7                        True       True
CD14                        True      False
CD19                       False  

## 5. Round-trip via CellOT env for anndata 0.7 compatibility (same as 01.5 §7)

In [11]:
files_to_convert = [
    os.path.join(DATASET_DIR, f"hvg_{flavor}_atlas_full.h5ad")
    for flavor in FLAVORS
]
v07_paths = []
for src in files_to_convert:
    dst = src.replace(".h5ad", "_v07.h5ad")
    if os.path.exists(dst):
        os.remove(dst)
    shutil.copy2(src, dst)
    v07_paths.append(dst)

print(f"Copied {len(v07_paths)} files. Stripping + rewriting via CellOT env ...")

strip_script = r"""
import sys, h5py
EMPTY = ["layers","obsm","obsp","uns","varm","varp"]
for p in sys.argv[1:]:
    with h5py.File(p, "r+") as f:
        for g in EMPTY:
            if g in f and len(f[g].keys())==0:
                del f[g]
        for a in ("encoding-type","encoding-version"):
            if a in f.attrs:
                del f.attrs[a]
"""

rewrite_script = r"""
import sys, os, h5py, numpy as np, pandas as pd, anndata as ad
from scipy import sparse
def _d(x): return x.decode() if isinstance(x,(bytes,np.bytes_)) else x
def load_obs(f):
    g = f["obs"]; idx = _d(g.attrs["_index"]) if "_index" in g.attrs else "index"
    index = [_d(x) for x in g[idx][:]]; cols={}
    for n in g.keys():
        if n==idx: continue
        node=g[n]
        if isinstance(node, h5py.Group) and "categories" in node and "codes" in node:
            cats=[_d(c) for c in node["categories"][:]]
            cols[n]=pd.Categorical.from_codes(node["codes"][:], categories=cats)
        else:
            arr=node[:]
            if arr.dtype.kind in ("O","S"):
                arr=np.array([_d(x) for x in arr])
            cols[n]=arr
    return pd.DataFrame(cols, index=pd.Index(index, name=idx))
def load_var(f):
    g = f["var"]; idx = _d(g.attrs["_index"]) if "_index" in g.attrs else "index"
    index = [_d(x) for x in g[idx][:]]
    cols={}
    for n in g.keys():
        if n==idx: continue
        node=g[n]
        arr = node[:]
        if arr.dtype.kind in ("O","S"):
            arr=np.array([_d(x) for x in arr])
        cols[n]=arr
    return pd.DataFrame(cols, index=pd.Index(index, name=idx))
def load_X(f):
    n=f["X"]
    if isinstance(n,h5py.Group):
        d=n["data"][:]; i=n["indices"][:]; p=n["indptr"][:]
        sh=tuple(n.attrs.get("shape", n.attrs.get("h5sparse_shape")))
        e=_d(n.attrs.get("encoding-type", b"csr_matrix"))
        return sparse.csc_matrix((d,i,p),shape=sh) if "csc" in e else sparse.csr_matrix((d,i,p),shape=sh)
    return n[:]
for p in sys.argv[1:]:
    with h5py.File(p,"r") as f:
        obs=load_obs(f); var=load_var(f); X=load_X(f)
    a=ad.AnnData(X=X,obs=obs,var=var)
    os.remove(p); a.write(p)
    print("rewrote",p,"shape",a.shape)
"""

subprocess.run([CELLOT_PY, "-c", strip_script, *v07_paths], check=True)
subprocess.run([CELLOT_PY, "-c", rewrite_script, *v07_paths], check=True)
print("\n_v07 files now CellOT-env compatible.")

Copied 2 files. Stripping + rewriting via CellOT env ...
rewrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_seurat_v3_atlas_full_v07.h5ad shape (8610, 1000)
rewrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_pearson_residuals_atlas_full_v07.h5ad shape (8610, 1000)

_v07 files now CellOT-env compatible.


## 6. Verify with CellOT env (anndata 0.7)

In [12]:
verify_script = r"""
import sys, json, numpy as np, anndata as ad
for p in sys.argv[1:]:
    a = ad.read(p)
    info = {"path": p, "shape": list(a.shape),
            "X_min": float(np.nanmin(a.X)), "X_max": float(np.nanmax(a.X)),
            "X_mean": float(np.nanmean(a.X)),
            "obs_cols": sorted(a.obs.columns.tolist()),
            "condition": a.obs["condition"].astype(str).value_counts().to_dict()}
    print(json.dumps(info))
"""
subprocess.run([CELLOT_PY, "-c", verify_script, *v07_paths], check=True)
print("\nVerified.")

{"path": "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_seurat_v3_atlas_full_v07.h5ad", "shape": [8610, 1000], "X_min": 0.0, "X_max": 8.464755058288574, "X_mean": 0.13639970123767853, "obs_cols": ["cell_type", "cell_type_ontology_term_id", "condition", "donor_id", "species", "tissue", "tissue_ontology_term_id"], "condition": {"mouse": 4305, "human": 4305}}
{"path": "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_pearson_residuals_atlas_full_v07.h5ad", "shape": [8610, 1000], "X_min": 0.0, "X_max": 8.464755058288574, "X_mean": 0.47790899872779846, "obs_cols": ["cell_type", "cell_type_ontology_term_id", "condition", "donor_id", "species", "tissue", "tissue_ontology_term_id"], "condition": {"mouse": 4305, "human": 4305}}

Verified.
